# House Price Prediction

**Dataset**: California Housing Price - `housing.csv`<br>

Kaggle link: https://www.kaggle.com/datasets/camnugent/california-housing

### 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from sklearn.metrics import(
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

In [8]:
# configuration
pd.set_option("display.max_columns", None)
sns.set_theme(style="darkgrid")

plt.rcParams.update({
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
})

RANDOM_STATE = 42
CSV_PATH = "housing.csv" # update path for a different dataset
Target_COL = "median_house_value" # target column name

### 2.Load Data

In [9]:
df = pd.read_csv(CSV_PATH)

In [10]:
print("DataFrame shape", df.shape)

DataFrame shape (20640, 10)


In [11]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


### 3.Exploratory Data Analysis(EDA)

**About:**<br>
EDA is the process of understanding your dataset before building a machine learning model.<br>

Goal of EDA:

* Understand data structure
* Find patterns
* Detect missing values
* Detect outliers
* Check relationships between features
* Decide preprocessing/feature engineering steps

In [12]:
df.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')

**Columns details:**
1. longitude: A measure of how far west a house is; a higher value is farther west<br>
2. latitude: A measure of how far north a house is; a higher value is farther north<br>
3. housingMedianAge: Median age of a house within a block; a lower number is a newer building<br>
4. totalRooms: Total number of rooms within a block<br>
5. totalBedrooms: Total number of bedrooms within a block<br>
6. population: Total number of people residing within a block<br>
7. households: Total number of households, a group of people residing within a home unit, for a block<br>
8. medianIncome: Median income for households within a block of houses (measured in tens of thousands of US Dollars)<br>
9. medianHouseValue: Median house value for households within a block (measured in US Dollars)<br>
10. oceanProximity: Location of the house w.r.t ocean/sea


In [13]:
# basic dataset overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [14]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Target_col:", Target_COL)
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Target_col: median_house_value
Numerical columns: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']
Categorical columns: ['ocean_proximity']


In [ ]:
# missing values analysis

print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


In [16]:
# check presence of encoded missing values
for col in df.columns:
    print(df[col].value_counts().head(20))

longitude
-118.31    162
-118.30    160
-118.29    148
-118.27    144
-118.32    142
-118.28    141
-118.35    140
-118.36    138
-118.19    135
-118.37    128
-118.25    128
-118.20    126
-118.14    125
-118.13    121
-118.26    121
-118.18    120
-118.34    119
-118.21    118
-118.15    116
-118.12    112
Name: count, dtype: int64
latitude
34.06    244
34.05    236
34.08    234
34.07    231
34.04    221
34.09    212
34.02    208
34.10    203
34.03    193
33.93    181
33.94    175
33.97    172
33.99    168
33.88    164
34.11    162
33.98    162
34.16    159
34.12    158
34.15    157
34.01    156
Name: count, dtype: int64
housing_median_age
52.0    1273
36.0     862
35.0     824
16.0     771
17.0     698
34.0     689
26.0     619
33.0     615
18.0     570
25.0     566
32.0     565
37.0     537
15.0     512
19.0     502
27.0     488
24.0     478
30.0     476
28.0     471
20.0     465
29.0     461
Name: count, dtype: int64
total_rooms
1527.0    18
1613.0    17
1582.0    17
2127.0    16


Note: 
* There are total 20,640 instances in the dataset.
* `total_bedrooms` attribute has only 20,433 null values, meaning that 207 districts are missing in this feature.
* All attributes/columns are numerical, except the `ocean_proximity`.

# More to Add!